In [32]:
import pandas as pd
import numpy as np

In [33]:
df=pd.read_csv('D:/Apna College/DAY-38/10) DateFruit Dataset.csv')

In [34]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [35]:
x=df.drop('Class',axis=1)
y=df['Class']

In [36]:
x.shape

(898, 34)

In [37]:
y.unique()

<StringArray>
['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY']
Length: 7, dtype: str

In [38]:
from sklearn.preprocessing import StandardScaler,LabelEncoder

In [39]:
coder=LabelEncoder()
y=coder.fit_transform(y)

In [40]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [41]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [42]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [43]:
X_train_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
y_trian_tensor=torch.tensor(y_train,dtype=torch.long)
X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)
train_dataset=TensorDataset(X_train_tensor,y_trian_tensor)
test_dataset=TensorDataset(X_test_tensor,y_test_tensor)
train_loader=DataLoader(train_dataset,shuffle=True,batch_size=32)
test_loader=DataLoader(test_dataset,batch_size=32)

In [46]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        self.model=nn.Sequential(
            nn.Linear(x.shape[1],64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,7)
        )
    def forward(self,x):
        return self.model(x)

In [47]:
model=ANN()
creteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [49]:
epochs=100
for epoch in range(epochs):
    model.train()
    running_loss=0.0
    for xb,yb in train_loader:
        optimizer.zero_grad()
        outputs=model(xb)
        loss=creteria(outputs,yb)
        loss.backward()
        optimizer.step()
        running_loss+=loss

    train_loss=running_loss / len(train_loader)
    print(f'Epochs{epoch+1}/{epochs},loss={train_loss}')

Epochs1/100,loss=0.030326824635267258
Epochs2/100,loss=0.025976134464144707
Epochs3/100,loss=0.02471112087368965
Epochs4/100,loss=0.022257216274738312
Epochs5/100,loss=0.022067591547966003
Epochs6/100,loss=0.029796691611409187
Epochs7/100,loss=0.030144918709993362
Epochs8/100,loss=0.029022833332419395
Epochs9/100,loss=0.021702367812395096
Epochs10/100,loss=0.022321954369544983
Epochs11/100,loss=0.024380553513765335
Epochs12/100,loss=0.029830019921064377
Epochs13/100,loss=0.025967000052332878
Epochs14/100,loss=0.019189639016985893
Epochs15/100,loss=0.017158277332782745
Epochs16/100,loss=0.015882935374975204
Epochs17/100,loss=0.016188517212867737
Epochs18/100,loss=0.019428838044404984
Epochs19/100,loss=0.019916601479053497
Epochs20/100,loss=0.016665851697325706
Epochs21/100,loss=0.015507889911532402
Epochs22/100,loss=0.015239537693560123
Epochs23/100,loss=0.017365822568535805
Epochs24/100,loss=0.014212215319275856
Epochs25/100,loss=0.012813828885555267
Epochs26/100,loss=0.013268506154417

In [57]:
model.eval()
correc=0
total=0
with torch.no_grad():
    for xa,ya in test_loader:
        output=model(xa)
        _,predictd=torch.max(output,1)
        correc+=(predictd==ya).sum().item()
        total+=ya.size(0)



In [59]:
correc/total

0.9388888888888889